In [4]:
# 비어있는 필드 개수 확인

import json
import pandas as pd

# 1. 확인할 필드 목록
TARGET_FIELDS = [
    'NODE_ID', 'IPRD_NM', 'PLCT_NM', 'NODE_TTLE', 'NODE_TTLE_EN', 'PBSH',
    'NODE_LINK', 'NODE_CLSS_01', 'NODE_CLSS_02', 'ABST_KR', 'ABST_EN',
    'AUTR_NM', 'KYWD'
]

# 파일 경로 (본인의 파일 경로로 수정)
file_path = 'SSU_Datathon2025_공학분야_62199_Final.json'

try:
    # 2. JSON 파일 읽기
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 3. 'NODE_LIST' 키 안에 있는 실제 데이터 리스트 꺼내기
    if "NODE_LIST" in data:
        df = pd.DataFrame(data["NODE_LIST"])
        
        # 4. 각 컬럼별 빈 문자열("") 개수 세기
        # (데이터프레임에 존재하는 컬럼에 대해서만 수행)
        valid_columns = [col for col in TARGET_FIELDS if col in df.columns]
        
        empty_counts = (df[valid_columns] == "").sum()
        
        print("--- 필드별 빈 값(\"\") 개수 ---")
        print(empty_counts)
        
        # (옵션) 전체 데이터 개수 대비 비율이 궁금하다면
        # print("\n--- 빈 값 비율 (%) ---")
        # print((empty_counts / len(df)) * 100)
        
    else:
        print("오류: JSON 파일 안에 'NODE_LIST'라는 키가 없습니다.")

except Exception as e:
    print(f"에러 발생: {e}")

--- 필드별 빈 값("") 개수 ---
NODE_ID             0
IPRD_NM             0
PLCT_NM             0
NODE_TTLE           0
NODE_TTLE_EN      554
PBSH                0
NODE_LINK           0
NODE_CLSS_01        0
NODE_CLSS_02        0
ABST_KR         14597
ABST_EN           744
AUTR_NM             0
KYWD              104
dtype: int64


In [1]:
# 논문 초록 한글/영어 둘다 없는 논문의 개수

import json
import pandas as pd

# 1. 파일 경로 (본인의 실제 파일 경로로 확인해주세요)
file_path = '../SSU_Datathon2025_공학분야_62199.json'

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 2. 데이터 프레임 변환
    if "NODE_LIST" in data:
        df = pd.DataFrame(data["NODE_LIST"])
        
        # -------------------------------------------------------
        # 요청 1: 수정된 필드명을 포함하여 각각의 빈 값 개수 확인
        # -------------------------------------------------------
        TARGET_FIELDS = [
            'NODE_ID', 'IPRD_NM', 'PLCT_NM', 'NODE_TTLE', 'NODE_TTLE_EN', 'PBSH',
            'NODE_LINK', 'NODE_CLSS_01', 'NODE_CLSS_02', 
            'ABST_KR', 'ABST_EN',  # <--- 수정된 필드명
            'AUTR_NM', 'KYWD'
        ]
        
        # 존재하는 컬럼만 추려서 검사
        valid_columns = [col for col in TARGET_FIELDS if col in df.columns]
        
        print("--- [1] 필드별 빈 값(\"\") 개수 ---")
        print((df[valid_columns] == "").sum())
        print("-" * 40)

        # -------------------------------------------------------
        # 요청 2: ABST_KR과 ABST_EN 둘 다 공백인 경우 확인
        # -------------------------------------------------------
        if 'ABST_KR' in df.columns and 'ABST_EN' in df.columns:
            # 두 조건이 모두 True인 행을 찾음 (& 연산자 사용)
            both_empty_df = df[(df['ABST_KR'] == "") & (df['ABST_EN'] == "")]
            count = len(both_empty_df)
            
            print(f"--- [2] 국문/영문 초록 둘 다 없는 논문 수 ---")
            print(f"개수: {count}개")
            
            # (선택 사항) 둘 다 없는 데이터의 ID만 보고 싶다면 아래 주석 해제
            # print("\n[해당 논문 ID 목록]")
            # print(both_empty_df['NODE_ID'].values)
            
        else:
            print("오류: 데이터에 ABST_KR 또는 ABST_EN 컬럼이 없습니다.")

    else:
        print("오류: JSON 파일 안에 'NODE_LIST' 키가 없습니다.")

except Exception as e:
    print(f"에러 발생: {e}")

--- [1] 필드별 빈 값("") 개수 ---
NODE_ID             0
IPRD_NM             0
PLCT_NM             0
NODE_TTLE           0
NODE_TTLE_EN    10650
PBSH                0
NODE_LINK           0
NODE_CLSS_01        0
NODE_CLSS_02        0
ABST_KR         35862
ABST_EN         16414
AUTR_NM             0
KYWD                0
dtype: int64
----------------------------------------
--- [2] 국문/영문 초록 둘 다 없는 논문 수 ---
개수: 15151개


In [3]:
# 논문 초록 한글/영어 둘 다 없는 논문 추출 후 csv로 저장
import json
import pandas as pd

# 1. 파일 경로 설정 (사용자 환경에 맞게 수정)
file_path = '../SSU_Datathon2025_공학분야_62199.json'
output_csv_name = 'empty_abstract_papers.csv'

try:
    # 2. JSON 파일 읽기
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if "NODE_LIST" in data:
        # 3. 데이터프레임 변환
        df = pd.DataFrame(data["NODE_LIST"])
        
        # 필드 존재 여부 확인 (안전장치)
        if 'ABST_KR' in df.columns and 'ABST_EN' in df.columns:
            
            # 4. 국문(ABST_KR)과 영문(ABST_EN)이 모두 빈 값("")인 행 필터링
            #    두 조건이 모두 참이어야 하므로 & 연산자 사용
            target_df = df[(df['ABST_KR'] == "") & (df['ABST_EN'] == "")]
            
            count = len(target_df)
            
            if count > 0:
                # 5. CSV 파일로 저장
                # encoding='utf-8-sig'는 엑셀에서 한글 깨짐을 방지하기 위함입니다.
                target_df.to_csv(output_csv_name, index=False, encoding='utf-8-sig')
                
                print(f"✅ 저장 완료!")
                print(f"- 파일명: {output_csv_name}")
                print(f"- 추출된 논문 수: {count}건")
            else:
                print("조건에 맞는(둘 다 빈 값인) 논문이 없습니다.")
                
        else:
            print("오류: 데이터프레임에 ABST_KR 또는 ABST_EN 컬럼이 없습니다.")
            
    else:
        print("오류: JSON 파일 내에 'NODE_LIST' 키가 없습니다.")

except FileNotFoundError:
    print(f"파일을 찾을 수 없습니다: {file_path}")
except Exception as e:
    print(f"에러 발생: {e}")

✅ 저장 완료!
- 파일명: empty_abstract_papers.csv
- 추출된 논문 수: 15151건


In [5]:
# 키워드 필드 자체가 없는 논문 개수

import json
import pandas as pd

# 1. 파일 경로
file_path = '../SSU_Datathon2025_공학분야_62199.json'

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if "NODE_LIST" in data:
        df = pd.DataFrame(data["NODE_LIST"])
        
        # 2. KYWD 컬럼이 존재하는지 확인
        if 'KYWD' in df.columns:
            # 3. KYWD 값이 NaN(누락)인 행 필터링
            missing_kywd_df = df[df['KYWD'].isna()]
            count = len(missing_kywd_df)
            
            print(f"--- 분석 결과 ---")
            print(f"KYWD 필드가 아예 없는(NaN) 논문 개수: {count}건")
            
            if count > 0:
                print("\n[샘플 NODE_ID (최대 5개)]")
                print(missing_kywd_df['NODE_ID'].head(5).tolist())
        else:
            # 컬럼 자체가 없으면 모든 데이터가 누락된 것
            print(f"⚠️ 'KYWD' 컬럼이 아예 없습니다. (총 {len(df)}건 모두 누락)")

    else:
        print("오류: 'NODE_LIST' 키를 찾을 수 없습니다.")

except Exception as e:
    print(f"에러 발생: {e}")

--- 분석 결과 ---
KYWD 필드가 아예 없는(NaN) 논문 개수: 16411건

[샘플 NODE_ID (최대 5개)]
['NODE10565029', 'NODE10565030', 'NODE10565031', 'NODE10561437', 'NODE10561438']


In [3]:
import json
import pandas as pd

# 1. 파일 경로 (사용자 환경에 맞게 수정)
file_path = '../SSU_Datathon2025_공학분야_62199.json'

try:
    # 데이터 로드
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if "NODE_LIST" in data:
        df = pd.DataFrame(data["NODE_LIST"])
        
        # 컬럼 존재 여부 확인
        has_kywd = 'KYWD' in df.columns
        has_kr = 'ABST_KR' in df.columns
        has_en = 'ABST_EN' in df.columns
        
        if has_kywd and has_kr and has_en:
            # -------------------------------------------------------------
            # [조건 필터링]
            # 1. KYWD는 NaN (데이터 없음)
            # 2. ABST_KR은 "" (빈 문자열)
            # 3. ABST_EN은 "" (빈 문자열)
            # -------------------------------------------------------------
            target_df = df[
                (df['KYWD'].isna()) & 
                (df['ABST_KR'] == "") & 
                (df['ABST_EN'] == "")
            ]
            
            count = len(target_df)
            
            print(f"--- 분석 결과 ---")
            print(f"조건: 키워드 없음(NaN) + 초록 둘 다 공백")
            print(f"해당되는 논문 개수: {count}건")
            
            if count > 0:
                # ---------------------------------------------------------
                # [추가된 부분] 결과를 CSV 파일로 저장
                # ---------------------------------------------------------
                output_filename = 'papers_no_keyword_no_abstract.csv'
                
                # encoding='utf-8-sig': 엑셀에서 열었을 때 한글 깨짐 방지
                target_df.to_csv(output_filename, index=False, encoding='utf-8-sig')
                
                print(f"\n[저장 완료] '{output_filename}' 파일에 저장되었습니다.")
                
                # (선택) 저장된 데이터의 일부 컬럼만 미리보기
                print("\n[저장된 데이터 미리보기 (상위 3개)]")
                print(target_df[['NODE_ID', 'NODE_TTLE']].head(3))
            
            else:
                print("\n조건에 맞는 논문이 없어 파일을 저장하지 않았습니다.")
                
        else:
            print("오류: 데이터프레임에 필수 컬럼(KYWD, ABST_KR, ABST_EN)이 누락되었습니다.")

    else:
        print("오류: 'NODE_LIST' 키를 찾을 수 없습니다.")

except Exception as e:
    print(f"에러 발생: {e}")

--- 분석 결과 ---
조건: 키워드 없음(NaN) + 초록 둘 다 공백
해당되는 논문 개수: 15089건

[저장 완료] 'papers_no_keyword_no_abstract.csv' 파일에 저장되었습니다.

[저장된 데이터 미리보기 (상위 3개)]
         NODE_ID                    NODE_TTLE
6   NODE10565029  기계설비 유지관리를 위한 건물관리시스템의 발전방향
7   NODE10565030   AI 및 IoT 기반 스마트 건물 자동제어시스템
22  NODE10561437    Submission Checklist etc.


In [1]:
# 원본 데이터에서 키워드 + 초록 둘 다 없는 논문 제거

import json
import pandas as pd
import os

# =============================================================================
# 1. 파일 경로 설정
# =============================================================================
# 처리할 입력 파일 (현재 작업 중인 최신 파일명을 넣으세요)
input_file_path = '../SSU_Datathon2025_공학분야_62199.json' 

# 삭제 후 저장할 최종 결과 파일명
output_file_path = 'SSU_Datathon2025_공학분야_62199_Except_No_KYWD_ABST.json'

def remove_empty_papers():
    print(f"📂 데이터 로드 중... ({input_file_path})")
    
    if not os.path.exists(input_file_path):
        print(f"❌ 오류: 파일({input_file_path})을 찾을 수 없습니다.")
        return

    try:
        # 1. JSON 파일 읽기
        with open(input_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # 데이터 구조 확인 및 DataFrame 변환
        if isinstance(data, dict) and "NODE_LIST" in data:
            df = pd.DataFrame(data["NODE_LIST"])
        elif isinstance(data, list):
            df = pd.DataFrame(data)
        else:
            print("❌ 데이터 구조를 인식할 수 없습니다.")
            return

        initial_count = len(df)
        
        # 2. 삭제 조건 설정
        # 조건: KYWD가 NaN(없음)이면서 AND 국문초록이 공백("") AND 영문초록이 공백("")
        # 주의: NaN 처리를 위해 isna() 사용
        remove_condition = (
            (df['KYWD'].isna() | (df['KYWD'] == "")) & 
            (df['ABST_KR'] == "") & 
            (df['ABST_EN'] == "")
        )
        
        # 3. 데이터 필터링 (조건에 해당하지 않는 것만 남김 -> ~ 기호 사용)
        cleaned_df = df[~remove_condition].copy()
        
        removed_count = initial_count - len(cleaned_df)
        
        print("-" * 50)
        print(f"📊 처리 결과")
        print(f"   - 원본 데이터 수: {initial_count:,}건")
        print(f"   - 제거된 데이터 수: {removed_count:,}건 (키워드X + 초록X)")
        print(f"   - 남은 데이터 수: {len(cleaned_df):,}건")
        print("-" * 50)

        if removed_count > 0:
            # 4. 저장 (URL 슬래시 보존을 위해 json 라이브러리 사용)
            # DataFrame -> 딕셔너리 리스트 변환
            # (JSON 저장 시 NaN이 있으면 에러가 날 수 있으므로 안전하게 처리)
            cleaned_data_list = cleaned_df.where(pd.notnull(cleaned_df), "").to_dict(orient='records')
            
            final_output = {"NODE_LIST": cleaned_data_list}
            
            with open(output_file_path, 'w', encoding='utf-8') as f:
                json.dump(final_output, f, ensure_ascii=False, indent=4)
                
            print(f"🎉 저장 완료! '{output_file_path}'")
            
            # (선택) 제거된 데이터들만 따로 CSV로 백업하고 싶다면 아래 주석 해제
            # removed_df = df[remove_condition]
            # removed_df.to_csv('removed_papers_backup.csv', index=False, encoding='utf-8-sig')
            # print(f"   (참고: 제거된 {removed_count}건은 'removed_papers_backup.csv'에 백업되었습니다.)")
            
        else:
            print("✨ 제거할 데이터가 없습니다. (조건에 맞는 논문 0건)")

    except Exception as e:
        print(f"❌ 에러 발생: {e}")

if __name__ == "__main__":
    remove_empty_papers()

📂 데이터 로드 중... (../SSU_Datathon2025_공학분야_62199.json)
--------------------------------------------------
📊 처리 결과
   - 원본 데이터 수: 62,199건
   - 제거된 데이터 수: 15,089건 (키워드X + 초록X)
   - 남은 데이터 수: 47,110건
--------------------------------------------------
🎉 저장 완료! 'SSU_Datathon2025_공학분야_62199_Except_No_KYWD_ABST.json'


In [7]:
# 키워드 필드는 없지만 한글 또는 영어 초록이 있는 논문 개수

import json
import pandas as pd

# 1. 파일 경로
file_path = '../SSU_Datathon2025_공학분야_62199.json'

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if "NODE_LIST" in data:
        df = pd.DataFrame(data["NODE_LIST"])
        
        # 필요한 컬럼 확인
        has_kywd = 'KYWD' in df.columns
        has_kr = 'ABST_KR' in df.columns
        has_en = 'ABST_EN' in df.columns
        
        if has_kywd and has_kr and has_en:
            # -------------------------------------------------------------
            # [조건 식]
            # 1. KYWD는 NaN (누락됨)
            #    AND (&)
            # 2. (국문 초록이 ""가 아님 OR 영문 초록이 ""가 아님)
            # -------------------------------------------------------------
            target_df = df[
                (df['KYWD'].isna()) & 
                ( (df['ABST_KR'] != "") | (df['ABST_EN'] != "") )
            ]
            
            count = len(target_df)
            
            print(f"--- 분석 결과 ---")
            print(f"조건: 키워드 없음(NaN) + 초록(한/영) 중 하나라도 있음")
            print(f"해당되는 논문 개수: {count}건")
            
            if count > 0:
                print("\n[샘플 NODE_ID (최대 5개)]")
                print(target_df['NODE_ID'].head(5).tolist())
                
                # (참고) 어떤 초록이 있는지 확인해보고 싶다면 아래 주석 해제
                # print("\n[샘플 데이터 초록 내용 확인]")
                # print(target_df[['NODE_ID', 'ABST_KR', 'ABST_EN']].head(5))
                
        else:
            print("오류: KYWD, ABST_KR, ABST_EN 중 없는 컬럼이 있어 분석할 수 없습니다.")

    else:
        print("오류: 'NODE_LIST' 키를 찾을 수 없습니다.")

except Exception as e:
    print(f"에러 발생: {e}")

--- 분석 결과 ---
조건: 키워드 없음(NaN) + 초록(한/영) 중 하나라도 있음
해당되는 논문 개수: 1322건

[샘플 NODE_ID (최대 5개)]
['NODE10565031', 'NODE10561439', 'NODE10561440', 'NODE10561441', 'NODE10561442']


In [3]:
import json
import pandas as pd

# 1. 파일 경로 설정
file_path = 'top_70_percent_corrected.json'
output_csv_name = 'keyword_missing_abstract_being_papers.csv'

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if "NODE_LIST" in data:
        df = pd.DataFrame(data["NODE_LIST"])
        
        # 컬럼 존재 여부 확인
        if {'KYWD', 'ABST_KR', 'ABST_EN'}.issubset(df.columns):
            
            # [전처리 1] NaN 값을 빈 문자열 ""로 통일 (비교 정확도 향상)
            df['KYWD'] = df['KYWD'].fillna("")
            df['ABST_KR'] = df['ABST_KR'].fillna("")
            df['ABST_EN'] = df['ABST_EN'].fillna("")

            # [전처리 2] 초록 내 줄바꿈(\n) 문자 제거 (CSV 저장 시 줄 밀림 방지)
            # 엑셀 깨짐의 주범인 줄바꿈과 탭 문자를 공백으로 치환
            df['ABST_KR'] = df['ABST_KR'].astype(str).replace(r'[\r\n\t]+', ' ', regex=True)
            df['ABST_EN'] = df['ABST_EN'].astype(str).replace(r'[\r\n\t]+', ' ', regex=True)

            # -------------------------------------------------------------
            # [수정된 조건 식]
            # 1. KYWD가 빈 문자열("")임
            #    AND
            # 2. (국문 초록이 ""가 아님 OR 영문 초록이 ""가 아님)
            # -------------------------------------------------------------
            target_df = df[
                (df['KYWD'] == "") & 
                ( (df['ABST_KR'] != "") | (df['ABST_EN'] != "") )
            ]
            
            count = len(target_df)
            
            print(f"--- 분석 결과 (줄바꿈/NaN 처리 완료) ---")
            print(f"조건: 키워드 없음 + 초록(한/영) 존재")
            print(f"해당되는 논문 개수: {count}건")
            
            if count > 0:
                # CSV 저장
                target_df.to_csv(output_csv_name, index=False, encoding='utf-8-sig')
                print(f"\n[저장 완료] '{output_csv_name}' 파일로 저장되었습니다.")
                print("이제 파일의 행 개수와 로그의 숫자가 정확히 일치할 것입니다.")
                
            else:
                print("\n조건에 맞는 데이터가 없습니다.")
                
        else:
            print("오류: 필요한 컬럼(KYWD, ABST_KR, ABST_EN)이 없습니다.")

    else:
        print("오류: 'NODE_LIST' 키를 찾을 수 없습니다.")

except Exception as e:
    print(f"에러 발생: {e}")

--- 분석 결과 (줄바꿈/NaN 처리 완료) ---
조건: 키워드 없음 + 초록(한/영) 존재
해당되는 논문 개수: 679건

[저장 완료] 'keyword_missing_abstract_being_papers.csv' 파일로 저장되었습니다.
이제 파일의 행 개수와 로그의 숫자가 정확히 일치할 것입니다.


In [1]:
import json
import pandas as pd
import os
import re

# -----------------------------------------------------------
# 1. 설정
# -----------------------------------------------------------
json_file_path = '../SSU_Datathon2025_공학분야_62199.json'

file_paths_if = {
    2021: '2021_인용지수_2년분.xls',
    2022: '2022_인용지수_2년분.xls',
    2023: '2023_인용지수_2년분.xls',
    2024: '2024_인용지수_2년분.xls'
}

# 사용자 수기 매핑 (이전에 확인한 내용)
manual_mapping = {
    "(사)한국CDE학회": "한국CDE학회",
    "ICT플랫폼학회": "아이씨티플랫폼학회",
    "유공압건설기계학회": "사단법인 유공압건설기계학회",
    "한국로봇학회(논문지)": "한국로봇학회",
    "한국염색가공학회": "한국염색가공학회",
    "한국위험물학회": "한국위험물학회",
    "한국자동차안전학회": "사단법인 한국자동차안전학회",
    "한국전자파학회JEES": "한국전자파학회",
    "한국정보통신학회JICCE": "한국정보통신학회",
    "한국컴퓨터그래픽스학회": "(사)한국컴퓨터그래픽스학회",
    "한국콘텐츠학회(IJOC)": "한국콘텐츠학회",
    "한국환경에너지공학회": "(사)한국환경에너지공학회"
}

# -----------------------------------------------------------
# 2. 유틸리티 & DB 로드
# -----------------------------------------------------------
def normalize_name(name):
    if pd.isna(name): return ""
    return re.sub(r'[^a-zA-Z0-9가-힣]', '', str(name).upper())

def load_if_database(file_paths):
    db = {}
    print("📂 인용지수 DB 로드 중...")
    for year, path in file_paths.items():
        if not os.path.exists(path):
            db[year] = set()
            continue
        try:
            df = pd.read_excel(path, engine='xlrd')
            df.columns = df.columns.str.replace('\n', '').str.strip()
            
            # 발행기관 컬럼 찾기
            org_cols = [c for c in df.columns if any(x in c for x in ['발행기관', '발행처', '학회'])]
            target_col = org_cols[0] if org_cols else (df.columns[1] if len(df.columns) > 1 else df.columns[0])
            
            # 정규화된 이름 집합 저장 (점수는 필요없고 존재 여부만 확인하면 됨)
            names_set = set()
            for name in df[target_col].dropna():
                names_set.add(normalize_name(name))
            db[year] = names_set
        except:
            db[year] = set()
    
    db[2025] = db.get(2024, set())
    return db

def main():
    if_db = load_if_database(file_paths_if)
    
    print(f"📂 JSON 로드 중... ({json_file_path})")
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        df = pd.DataFrame(data["NODE_LIST"])
        df['temp_Year'] = df['PBSH'].astype(str).str.strip().str[:4]
        
        # -----------------------------------------------------------
        # 3. 매칭 여부 판단
        # -----------------------------------------------------------
        print("🔍 매칭 상태 전수 조사 중...")
        
        results = []
        still_unmatched_societies = set()
        
        for idx, row in df.iterrows():
            try:
                y = int(row['temp_Year'])
            except:
                y = 0
            
            original_name = row['IPRD_NM']
            category = row['NODE_CLSS_02']
            
            # 1. 수기 매핑 적용
            if original_name in manual_mapping:
                search_name = manual_mapping[original_name]
                is_manual_fixed = True
            else:
                search_name = original_name
                is_manual_fixed = False
            
            # 2. DB 확인
            norm_name = normalize_name(search_name)
            valid_set = if_db.get(y, set())
            
            is_matched = norm_name in valid_set
            
            if not is_matched:
                still_unmatched_societies.add(original_name)
            
            results.append({
                'Year': y,
                'Category': category,
                'Society': original_name,
                'Is_Matched': is_matched
            })
            
        result_df = pd.DataFrame(results)
        
        # -----------------------------------------------------------
        # 4. 통계 산출
        # -----------------------------------------------------------
        total_papers = len(result_df)
        matched_papers = result_df['Is_Matched'].sum()
        unmatched_papers = total_papers - matched_papers
        
        print("\n" + "="*50)
        print("📊 [최종 분석 결과] 수기 보정 적용 후")
        print("="*50)
        print(f"1. 전체 논문 수: {total_papers:,}건")
        print(f"2. 매칭 성공:    {matched_papers:,}건 ({matched_papers/total_papers*100:.2f}%)")
        print(f"3. 매칭 실패:    {unmatched_papers:,}건 ({unmatched_papers/total_papers*100:.2f}%)")
        print("-" * 50)
        
        # 분야별 실패율
        print("\n[분야별 미매칭 비율]")
        summary = result_df.groupby('Category')['Is_Matched'].agg(['count', 'sum'])
        summary.columns = ['Total', 'Matched']
        summary['Unmatched'] = summary['Total'] - summary['Matched']
        summary['Fail_Rate(%)'] = (summary['Unmatched'] / summary['Total'] * 100).round(2)
        
        # 실패율 높은 순 정렬
        summary = summary.sort_values('Fail_Rate(%)', ascending=False)
        print(summary[['Total', 'Unmatched', 'Fail_Rate(%)']])
        
        # -----------------------------------------------------------
        # 5. 파일 저장
        # -----------------------------------------------------------
        # 1) 통계 파일
        summary.to_csv('match_fail_statistics.csv', encoding='utf-8-sig')
        
        # 2) 여전히 매칭 안 되는 학회 명단
        if still_unmatched_societies:
            unmatched_list_df = pd.DataFrame(sorted(list(still_unmatched_societies)), columns=['Society_Name'])
            unmatched_list_df.to_csv('still_unmatched_societies.csv', index=False, encoding='utf-8-sig')
            print(f"\n📂 남은 미매칭 학회 명단 저장됨: still_unmatched_societies.csv ({len(still_unmatched_societies)}개)")
        
    except Exception as e:
        print(f"에러 발생: {e}")

if __name__ == "__main__":
    main()

📂 인용지수 DB 로드 중...
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133824, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133312, actual size 512
📂 JSON 로드 중... (../SSU_Datathon2025_공학분야_62199.json)
🔍 매칭 상태 전수 조사 중...

📊 [최종 분석 결과] 수기 보정 적용 후
1. 전체 논문 수: 62,199건
2. 매칭 성공:    58,112건 (93.43%)
3. 매칭 실패:    4,087건 (6.57%)
--------------------------------------------------

[분야별 미매칭 비율]
          Total  Unmatched  Fail_Rate(%)
Category                                
컴퓨터학       6831       2141         31.34
기타 공학      1984        325         16.38
조선해양공학     1186        143         12.06
기계공학      12666        822          6.49
건축공학      11464        451          3.93
재료·에너지공학   1408         26          1.85
전기전자공학    18255        179          0.98
공학 일반      5569          0          0.00
산업공학       1392          0          0.00
화학공학       

In [2]:
import json
import pandas as pd
import os
import re

# -----------------------------------------------------------
# 1. 설정 및 수기 매핑
# -----------------------------------------------------------
json_file_path = '../SSU_Datathon2025_공학분야_62199.json'

file_paths_if = {
    2021: '2021_인용지수_2년분.xls',
    2022: '2022_인용지수_2년분.xls',
    2023: '2023_인용지수_2년분.xls',
    2024: '2024_인용지수_2년분.xls'
}

# 사용자 수기 매핑 적용
manual_mapping = {
    "(사)한국CDE학회": "한국CDE학회",
    "ICT플랫폼학회": "아이씨티플랫폼학회",
    "유공압건설기계학회": "사단법인 유공압건설기계학회",
    "한국로봇학회(논문지)": "한국로봇학회",
    "한국염색가공학회": "한국염색가공학회",
    "한국위험물학회": "한국위험물학회",
    "한국자동차안전학회": "사단법인 한국자동차안전학회",
    "한국전자파학회JEES": "한국전자파학회",
    "한국정보통신학회JICCE": "한국정보통신학회",
    "한국컴퓨터그래픽스학회": "(사)한국컴퓨터그래픽스학회",
    "한국콘텐츠학회(IJOC)": "한국콘텐츠학회",
    "한국환경에너지공학회": "(사)한국환경에너지공학회"
}

# -----------------------------------------------------------
# 2. 유틸리티 & DB 로드
# -----------------------------------------------------------
def normalize_name(name):
    if pd.isna(name): return ""
    return re.sub(r'[^a-zA-Z0-9가-힣]', '', str(name).upper())

def load_if_database(file_paths):
    db = {}
    print("📂 인용지수 DB 로드 중...")
    for year, path in file_paths.items():
        if not os.path.exists(path):
            db[year] = set()
            continue
        try:
            df = pd.read_excel(path, engine='xlrd')
            df.columns = df.columns.str.replace('\n', '').str.strip()
            
            # 발행기관 컬럼 찾기
            org_cols = [c for c in df.columns if any(x in c for x in ['발행기관', '발행처', '학회'])]
            target_col = org_cols[0] if org_cols else (df.columns[1] if len(df.columns) > 1 else df.columns[0])
            
            # 정규화된 이름 집합 저장
            names_set = set()
            for name in df[target_col].dropna():
                names_set.add(normalize_name(name))
            db[year] = names_set
        except:
            db[year] = set()
    
    db[2025] = db.get(2024, set())
    return db

def main():
    # 1. DB 로드
    if_db = load_if_database(file_paths_if)
    
    # 2. JSON 로드
    print(f"📂 JSON 파일 로드 중... ({json_file_path})")
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        df = pd.DataFrame(data["NODE_LIST"])
        df['temp_Year'] = df['PBSH'].astype(str).str.strip().str[:4]
        
        # -----------------------------------------------------------
        # 3. [핵심] 키워드 & 초록(한/영) 모두 없는 논문 필터링
        # -----------------------------------------------------------
        print("🔍 조건 필터링: [키워드 X] AND [국문초록 X] AND [영문초록 X]")
        
        # 빈 값 조건: NaN이거나, 빈 문자열("")이거나, 공백만 있는 경우
        cond_kywd = (df['KYWD'].isna()) | (df['KYWD'].astype(str).str.strip() == "")
        cond_abst_kr = (df['ABST_KR'].isna()) | (df['ABST_KR'].astype(str).str.strip() == "")
        cond_abst_en = (df['ABST_EN'].isna()) | (df['ABST_EN'].astype(str).str.strip() == "")
        
        target_df = df[cond_kywd & cond_abst_kr & cond_abst_en].copy()
        
        if len(target_df) == 0:
            print("조건에 맞는 논문이 없습니다.")
            return

        print(f"   => 대상 논문 수: {len(target_df):,}건 (전체 {len(df):,}건 중)")

        # -----------------------------------------------------------
        # 4. 매칭율 분석
        # -----------------------------------------------------------
        matched_count = 0
        unmatched_count = 0
        unmatched_societies = set()
        
        for idx, row in target_df.iterrows():
            try:
                y = int(row['temp_Year'])
            except:
                y = 0
            
            original_name = row['IPRD_NM']
            
            # 수기 보정 적용
            if original_name in manual_mapping:
                search_name = manual_mapping[original_name]
            else:
                search_name = original_name
            
            # 매칭 확인
            norm_name = normalize_name(search_name)
            valid_set = if_db.get(y, set())
            
            if norm_name in valid_set:
                matched_count += 1
            else:
                unmatched_count += 1
                unmatched_societies.add(original_name)
        
        # -----------------------------------------------------------
        # 5. 결과 출력
        # -----------------------------------------------------------
        print("\n" + "="*60)
        print("📊 [키워드/초록 누락 논문] 매칭 분석 결과")
        print("="*60)
        print(f"1. 대상 논문 수 (A): {len(target_df):,}건")
        print(f"2. 매칭 성공 (B):    {matched_count:,}건")
        print(f"3. 매칭 실패 (C):    {unmatched_count:,}건")
        print("-" * 60)
        
        fail_rate = (unmatched_count / len(target_df)) * 100
        print(f"🔴 매칭 실패율 (C/A): {fail_rate:.2f}%")
        print("="*60)
        
        print(f"\n[참고] 이 집단에서 여전히 매칭 안 되는 학회들 (총 {len(unmatched_societies)}개)")
        # 너무 많으면 일부만 출력
        sorted_list = sorted(list(unmatched_societies))
        for name in sorted_list[:10]:
            print(f"- {name}")
        if len(sorted_list) > 10:
            print(f"... 외 {len(sorted_list)-10}개")

        # CSV 저장 (필요시 사용)
        pd.DataFrame(sorted_list, columns=['Unmatched_Society']).to_csv('empty_paper_unmatched_list.csv', index=False, encoding='utf-8-sig')
        print(f"\n📂 미매칭 학회 목록 저장됨: empty_paper_unmatched_list.csv")

    except Exception as e:
        print(f"에러 발생: {e}")

if __name__ == "__main__":
    main()

📂 인용지수 DB 로드 중...
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133824, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133312, actual size 512
📂 JSON 파일 로드 중... (../SSU_Datathon2025_공학분야_62199.json)
🔍 조건 필터링: [키워드 X] AND [국문초록 X] AND [영문초록 X]
   => 대상 논문 수: 15,089건 (전체 62,199건 중)

📊 [키워드/초록 누락 논문] 매칭 분석 결과
1. 대상 논문 수 (A): 15,089건
2. 매칭 성공 (B):    13,596건
3. 매칭 실패 (C):    1,493건
------------------------------------------------------------
🔴 매칭 실패율 (C/A): 9.89%

[참고] 이 집단에서 여전히 매칭 안 되는 학회들 (총 11개)
- Korean Institute of Information Scientists and Engineers
- Korean Society for Precision Engineering
- 새건축사협의회
- 서울대학교 환경대학원
- 한국게임학회
- 한국기계연구원
- 한국위험물학회
- 한국전지학회
- 한국항공협회
- 한국해양환경·에너지학회
... 외 1개

📂 미매칭 학회 목록 저장됨: empty_paper_unmatched_list.csv
